In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os
import yaml
import torch

BASE_PATH = "/content/drive/MyDrive/mosquito-robustness"

with open(os.path.join(BASE_PATH, "configs/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

Using device: cuda


In [16]:

import numpy as np
import random

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


#loading
import os
from torchvision import datasets

!unzip -q "/content/drive/MyDrive/images.zip" -d "/content/"

base_dir = "/content/images"

train_dir = os.path.join(base_dir, "training")
test_dir  = os.path.join(base_dir, "testing")

print("Train classes:", sorted(os.listdir(train_dir)))
print("Test classes:", sorted(os.listdir(test_dir)))

full_train_dataset = datasets.ImageFolder(train_dir)
test_dataset_raw   = datasets.ImageFolder(test_dir)

print("Train samples:", len(full_train_dataset))
print("Test samples:", len(test_dataset_raw))
print("Classes:", full_train_dataset.classes)


#val split
from torch.utils.data import random_split

val_size = int(0.1 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size

train_dataset_raw, val_dataset_raw = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

print("Train:", len(train_dataset_raw))
print("Val:", len(val_dataset_raw))
print("Test:", len(test_dataset_raw))


#transforms
from torchvision import transforms

image_size = config["data"]["image_size"]

train_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),  # deterministic due to seed
    transforms.ToTensor(),
    transforms.Normalize(
        config["normalization"]["mean"],
        config["normalization"]["std"]
    )
])

test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        config["normalization"]["mean"],
        config["normalization"]["std"]
    )
])


#wrapdataset
class TransformedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y


train_dataset = TransformedDataset(train_dataset_raw, train_transforms)
val_dataset   = TransformedDataset(val_dataset_raw, test_transforms)
test_dataset  = TransformedDataset(test_dataset_raw, test_transforms)


#dataloaders
from torch.utils.data import DataLoader

batch_size = config["training"]["batch_size"]

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    worker_init_fn=seed_worker,
    generator=g
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    worker_init_fn=seed_worker,
    generator=g
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    worker_init_fn=seed_worker,
    generator=g
)

print("✅ DataLoaders ready (deterministic).")

replace /content/images/testing/Aedes Aegypti/Aedes Aegypti_401.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Train classes: ['Aedes Aegypti', 'Aedes Albopictus', 'Anopheles Albimanus', 'Anopheles Arabiensis', 'Anopheles Freeborni', 'Culex Quinquefasciatus']
Test classes: ['Aedes Aegypti', 'Aedes Albopictus', 'Anopheles Albimanus', 'Anopheles Arabiensis', 'Anopheles Freeborni', 'Culex Quinquefasciatus']
Train samples: 2400
Test samples: 600
Classes: ['Aedes Aegypti', 'Aedes Albopictus', 'Anopheles Albimanus', 'Anopheles Arabiensis', 'Anopheles Freeborni', 'Culex Quinquefasciatus']
Train: 2160
Val: 240
Test: 600
✅ DataLoaders ready (deterministic).


In [17]:
# eff_model, res_model, swin_model already defined (NOTEBOOK 3 CODES)



import torch.nn as nn
from torchvision import models

!unzip -q "/content/drive/MyDrive/images.zip" -d "/content/"
train_dir = "/content/images/training"
num_classes = len(os.listdir(train_dir))

print("Number of classes:", num_classes)

#efficientnet
def get_efficientnet(num_classes):
    model = models.efficientnet_b0(weights="IMAGENET1K_V1")

    in_features = model.classifier[1].in_features

    # Replace classifier
    model.classifier[1] = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for param in model.features.parameters():
        param.requires_grad = False

    return model, in_features


def get_resnet(num_classes):
    model = models.resnet50(weights="IMAGENET1K_V1")

    in_features = model.fc.in_features

    model.fc = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad = False

    return model, in_features


def get_swin(num_classes):
    model = models.swin_v2_t(weights="IMAGENET1K_V1")

    in_features = model.head.in_features

    model.head = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = False

    return model, in_features


class ModelWithEmbedding(nn.Module):
    def __init__(self, model, model_name):
        super().__init__()
        self.model = model
        self.model_name = model_name

    def forward(self, x):
        if self.model_name == "resnet":
            # Extract features before FC
            features = self.model.avgpool(self.model.layer4(
                        self.model.layer3(
                        self.model.layer2(
                        self.model.layer1(
                        self.model.maxpool(
                        self.model.relu(
                        self.model.bn1(
                        self.model.conv1(x)))))))))
            features = torch.flatten(features, 1)
            logits = self.model.fc(features)

        elif self.model_name == "efficientnet":
            features = self.model.features(x)
            features = self.model.avgpool(features)
            features = torch.flatten(features, 1)
            logits = self.model.classifier(features)

        elif self.model_name == "swin":

            x=self.model.features(x)

            x=self.model.norm(x)

            features = x.mean(dim=(1,2))

            logits = self.model.head(features)
        return features, logits


eff_model, eff_dim = get_efficientnet(num_classes)
res_model, res_dim = get_resnet(num_classes)
swin_model, swin_dim = get_swin(num_classes)

eff_model = ModelWithEmbedding(eff_model, "efficientnet").to(DEVICE)
res_model = ModelWithEmbedding(res_model, "resnet").to(DEVICE)
swin_model = ModelWithEmbedding(swin_model, "swin").to(DEVICE)

print("Models initialized.")




Using device: cuda
replace /content/images/testing/Aedes Aegypti/Aedes Aegypti_401.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Number of classes: 6
Models initialized.


In [18]:


criterion = nn.CrossEntropyLoss()

def get_optimizer(model):
    return torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config["training"]["learning_rate"]
    )

In [19]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        _, logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    return total_loss / len(loader), acc

In [20]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            _, logits = model(images)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [21]:
def train_model(model, name):
    print(f"\nTraining {name}...")

    optimizer = get_optimizer(model)
    epochs = config["training"]["epochs"]

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        val_acc = evaluate(model, val_loader)

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Loss: {train_loss:.4f} "
              f"Train Acc: {train_acc:.4f} "
              f"Val Acc: {val_acc:.4f}")

    return model

In [22]:
eff_model = train_model(eff_model, "EfficientNet")
res_model = train_model(res_model, "ResNet")
swin_model = train_model(swin_model, "Swin")


Training EfficientNet...
Epoch [1/20] Loss: 1.6539 Train Acc: 0.3722 Val Acc: 0.6208
Epoch [2/20] Loss: 1.3954 Train Acc: 0.6153 Val Acc: 0.7667
Epoch [3/20] Loss: 1.2023 Train Acc: 0.7282 Val Acc: 0.7917
Epoch [4/20] Loss: 1.0631 Train Acc: 0.7727 Val Acc: 0.8292
Epoch [5/20] Loss: 0.9503 Train Acc: 0.8065 Val Acc: 0.8667
Epoch [6/20] Loss: 0.8715 Train Acc: 0.8282 Val Acc: 0.8583
Epoch [7/20] Loss: 0.8080 Train Acc: 0.8282 Val Acc: 0.8625
Epoch [8/20] Loss: 0.7397 Train Acc: 0.8481 Val Acc: 0.8958
Epoch [9/20] Loss: 0.6989 Train Acc: 0.8523 Val Acc: 0.8833
Epoch [10/20] Loss: 0.6624 Train Acc: 0.8579 Val Acc: 0.8917
Epoch [11/20] Loss: 0.6322 Train Acc: 0.8694 Val Acc: 0.9000
Epoch [12/20] Loss: 0.6027 Train Acc: 0.8694 Val Acc: 0.8875
Epoch [13/20] Loss: 0.5643 Train Acc: 0.8699 Val Acc: 0.9083
Epoch [14/20] Loss: 0.5551 Train Acc: 0.8801 Val Acc: 0.8792
Epoch [15/20] Loss: 0.5113 Train Acc: 0.8917 Val Acc: 0.9000
Epoch [16/20] Loss: 0.4973 Train Acc: 0.8907 Val Acc: 0.8958
Epoch [

In [23]:
model_dir = os.path.join(BASE_PATH, "models")

torch.save(eff_model.state_dict(), os.path.join(model_dir, "efficientnet.pth"))
torch.save(res_model.state_dict(), os.path.join(model_dir, "resnet.pth"))
torch.save(swin_model.state_dict(), os.path.join(model_dir, "swin.pth"))

print("Models saved.")

Models saved.
